# EXP-19: Light GPU Hybrid — BERTimbau Otimizado + TF-IDF Stacking

**Competition:** SPR 2026 — Mammography Report Classification  
**Task:** Classify mammography reports (Portuguese) into BI-RADS 0–6  
**Metric:** Macro F1  
**Runtime:** GPU leve (~20-30 min)  

## Estrategia baseada nas licoes dos exp01-18:
1. **BERTimbau Base** com TODOS os truques comprovados:
   - FGM adversarial training (+0.5-1% F1)
   - Weighted Focal Loss (gamma=2.0) para classes raras
   - Multi-sample dropout head (regularizacao + calibracao)
   - Cosine schedule com warmup
   - **Apenas 3 epochs** (leve, rapido, suficiente)
2. **TF-IDF + SVC** (melhor modelo CPU dos experimentos anteriores)
3. **TF-IDF + LightGBM** com features clinicas enriquecidas
4. **CatBoost** com SVD + dense features
5. **Meta-learner stacking** (4 modelos → LR meta)
6. **Threshold tuning** + guardrails clinicos

**Setup Kaggle:** Adicionar datasets:
- `cortese/bert-base-portuguese-cased`
- `spr-2026-mammography-report-classification`

In [ ]:
import os, gc, re, glob, time, hashlib, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack, csr_matrix
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
except ImportError:
    !pip install -q lightgbm
    import lightgbm as lgb

try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip install -q catboost
    from catboost import CatBoostClassifier

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB' if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else '')

# --- Paths ---
def find_input(pattern):
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if pattern in d.lower():
                return os.path.join(root, d)
    hits = glob.glob(f'/kaggle/input/**/*{pattern}*', recursive=True)
    return hits[0] if hits else None

MODEL_PATH = find_input('bert-base-portuguese-cased')
COMP_DIR   = find_input('spr-2026-mammography')
assert MODEL_PATH, f'BERTimbau not found! {os.listdir("/kaggle/input")}'
assert COMP_DIR,   f'Competition not found! {os.listdir("/kaggle/input")}'

train_df = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))

# --- Config ---
N_CLASSES    = 7
N_FOLDS      = 4        # 4 folds para economizar GPU time
MAX_LEN      = 256
BERT_BS      = 32
BERT_EPOCHS  = 3        # LEVE: apenas 3 epochs
BERT_LR      = 2e-5
WARMUP_RATIO = 0.1
WD           = 0.01
FOCAL_GAMMA  = 2.0
FGM_EPS      = 0.5
MULTI_DROP_N = 5
MULTI_DROP_P = 0.3
SEED         = 42

print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Class distribution:\n{train_df["target"].value_counts().sort_index()}')

In [ ]:
# =============================================================================
# Cell 2: Text Preprocessing + Groups
# =============================================================================

def clean_text_light(text):
    """Light cleaning for BERT (preserves structure)."""
    if pd.isna(text): return ''
    text = str(text).lower().strip()
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)
    return text

def clean_text_tfidf(s):
    """Aggressive cleaning for TF-IDF."""
    s = clean_text_light(s)
    s = re.sub(r'\b(cm|mm)\b', ' \\1 ', s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def extract_achados(s):
    match = re.search(r'achados[:\s]*(.*?)(?:impress[aã]o|conclus[aã]o|an[aá]lise comparativa|opini[aã]o|$)', s, flags=re.DOTALL)
    return match.group(1).strip() if match else ""

def stable_hash(s):
    return hashlib.md5(str(s).encode('utf-8')).hexdigest()

for df in [train_df, test_df]:
    df['clean']       = df['report'].apply(clean_text_light)
    df['clean_tfidf'] = df['report'].apply(clean_text_tfidf)
    df['achados']     = df['clean'].apply(extract_achados)
    df['achados_tfidf'] = df['achados'].apply(clean_text_tfidf)

groups = train_df['clean'].apply(stable_hash).values
labels = train_df['target'].values

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(train_df, labels, groups))

print(f'Unique groups: {len(set(groups))} / {len(groups)}')

In [ ]:
# =============================================================================
# Cell 3: Dense Clinical Features
# =============================================================================

def extract_birads_number(text):
    match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    return int(match.group(1)) if match else -1

def extract_dense_features(df):
    feat = pd.DataFrame(index=df.index)
    text = df['clean']
    
    feat['report_length']    = text.str.len()
    feat['word_count']       = text.str.split().str.len().fillna(0).astype(int)
    feat['line_count']       = text.str.count('\n') + 1
    feat['achados_length']   = df['achados'].str.len()
    feat['has_measurement']  = text.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    feat['has_spiculation']  = text.str.contains(r'espiculad', regex=True).astype(int)
    feat['has_distortion']   = text.str.contains(r'distor[cç][aã]o arquitetural', regex=True).astype(int)
    feat['has_biopsy']       = text.str.contains(r'bi[oó]psia|carcinoma|resultado de cine', regex=True).astype(int)
    feat['has_nodule']       = text.str.contains(r'n[oó]dulo', regex=True).astype(int)
    feat['has_calcification'] = text.str.contains(r'calcifica[cç]', regex=True).astype(int)
    feat['has_microcalc']    = text.str.contains(r'microcalcifica', regex=True).astype(int)
    feat['has_asymmetry']    = text.str.contains(r'assimetria', regex=True).astype(int)
    feat['has_suspicious']   = text.str.contains(r'suspeit[oa]|malign[oa]', regex=True).astype(int)
    feat['has_benign']       = text.str.contains(r'benign[oa]|cisto[s]?\b', regex=True).astype(int)
    feat['has_irregular']    = text.str.contains(r'irregular', regex=True).astype(int)
    feat['has_nodulo_espic'] = text.str.contains(r'n[oó]dulo\s+espiculad', regex=True).astype(int)
    feat['has_birads']       = text.str.contains(r'bi-?rads|categoria\s*\d', regex=True).astype(int)
    feat['birads_mentioned'] = text.apply(extract_birads_number)
    feat['risk_score'] = (
        feat['has_spiculation'] * 3 + feat['has_distortion'] * 3 +
        feat['has_nodulo_espic'] * 4 + feat['has_biopsy'] * 5 +
        feat['has_suspicious'] * 3 + feat['has_irregular'] * 2 +
        feat['has_microcalc'] * 2 + feat['has_asymmetry'] * 2 -
        feat['has_benign'] * 2
    )
    return feat

dense_train = extract_dense_features(train_df)
dense_test  = extract_dense_features(test_df)
print(f'Dense features: {dense_train.shape[1]}')

In [ ]:
# =============================================================================
# Cell 4: BERT Model Definition (FGM + Focal Loss + Multi-Dropout)
# =============================================================================

class ReportDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class MultiSampleDropoutHead(nn.Module):
    def __init__(self, hidden_size, num_labels, n_drops=5, drop_p=0.3):
        super().__init__()
        self.dropouts = nn.ModuleList([nn.Dropout(drop_p) for _ in range(n_drops)])
        self.classifier = nn.Linear(hidden_size, num_labels)
    def forward(self, hidden_state):
        if self.training:
            return torch.stack([self.classifier(d(hidden_state)) for d in self.dropouts]).mean(0)
        return self.classifier(hidden_state)

class BERTClassifier(nn.Module):
    def __init__(self, model_path, num_labels, n_drops=5, drop_p=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_path)
        self.head = MultiSampleDropoutHead(self.bert.config.hidden_size, num_labels, n_drops, drop_p)
    def forward(self, input_ids, attention_mask, token_type_ids=None, **kw):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        return self.head(out.last_hidden_state[:, 0])  # CLS token

class FGM:
    def __init__(self, model, eps=0.5, emb_name='word_embeddings'):
        self.model, self.eps, self.emb_name, self.backup = model, eps, emb_name, {}
    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0: param.data.add_(self.eps * param.grad / norm)
    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup: param.data = self.backup[name]
        self.backup = {}

class WeightedFocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer('alpha', alpha)
        self.gamma = gamma
    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, reduction='none')
        pt = torch.exp(-ce)
        return (self.alpha[labels] * ((1 - pt) ** self.gamma) * ce).mean()

def compute_class_weights(labels, num_classes=7):
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    weights = len(labels) / (num_classes * counts + 1e-6)
    return torch.tensor(weights / weights.mean(), dtype=torch.float32)

print('BERT components defined.')

In [ ]:
# =============================================================================
# Cell 5: BERT Training Loop
# =============================================================================

def train_bert_fold(tr_idx, va_idx, fold):
    print(f'\n{"="*40} BERT FOLD {fold} {"="*40}')
    tr_texts = train_df.iloc[tr_idx]['report'].tolist()
    va_texts = train_df.iloc[va_idx]['report'].tolist()
    tr_labels = labels[tr_idx]
    va_labels = labels[va_idx]
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    tr_enc = tokenizer(tr_texts, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
    va_enc = tokenizer(va_texts, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
    
    tr_loader = DataLoader(ReportDataset(tr_enc, tr_labels), batch_size=BERT_BS, shuffle=True, num_workers=2, pin_memory=True)
    va_loader = DataLoader(ReportDataset(va_enc), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    
    model = BERTClassifier(MODEL_PATH, N_CLASSES, MULTI_DROP_N, MULTI_DROP_P).to(DEVICE)
    alpha = compute_class_weights(tr_labels).to(DEVICE)
    criterion = WeightedFocalLoss(alpha, gamma=FOCAL_GAMMA)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=BERT_LR, weight_decay=WD)
    total_steps = len(tr_loader) * BERT_EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    fgm = FGM(model, eps=FGM_EPS)
    scaler = torch.cuda.amp.GradScaler()
    
    best_f1, best_oof, best_state = 0, None, None
    
    for epoch in range(BERT_EPOCHS):
        model.train()
        total_loss = 0
        for batch in tr_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            tids = batch.get('token_type_ids', torch.zeros_like(ids)).to(DEVICE)
            lbl = batch['labels'].to(DEVICE)
            
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                loss = criterion(model(ids, mask, tids), lbl)
            scaler.scale(loss).backward()
            
            # FGM adversarial
            fgm.attack()
            with torch.cuda.amp.autocast():
                loss_adv = criterion(model(ids, mask, tids), lbl)
            scaler.scale(loss_adv).backward()
            fgm.restore()
            
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
        
        # Validation
        model.eval()
        all_probs = []
        with torch.no_grad():
            for batch in va_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                tids = batch.get('token_type_ids', torch.zeros_like(ids)).to(DEVICE)
                with torch.cuda.amp.autocast():
                    logits = model(ids, mask, tids)
                all_probs.append(F.softmax(logits, dim=-1).cpu().numpy())
        
        oof_probs = np.vstack(all_probs)
        f1 = f1_score(va_labels, oof_probs.argmax(-1), average='macro')
        print(f'  Epoch {epoch} | loss={total_loss/len(tr_loader):.4f} | val_f1={f1:.4f}')
        
        if f1 > best_f1:
            best_f1, best_oof = f1, oof_probs.copy()
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    print(f'  Best val F1: {best_f1:.4f}')
    
    # Test predictions
    model.load_state_dict(best_state)
    model.eval()
    test_enc = tokenizer(test_df['report'].tolist(), truncation=True, padding='max_length',
                         max_length=MAX_LEN, return_tensors='pt')
    test_loader = DataLoader(ReportDataset(test_enc), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    
    test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            tids = batch.get('token_type_ids', torch.zeros_like(ids)).to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(ids, mask, tids)
            test_probs.append(F.softmax(logits, dim=-1).cpu().numpy())
    
    del model, optimizer, scheduler, scaler, fgm, criterion
    gc.collect(); torch.cuda.empty_cache()
    
    return best_oof, np.vstack(test_probs), best_f1

print('Training function defined.')

In [ ]:
# =============================================================================
# Cell 6: Run BERT CV
# =============================================================================

t0 = time.time()

bert_oof  = np.zeros((len(train_df), N_CLASSES))
bert_test = np.zeros((len(test_df), N_CLASSES))
bert_scores = []

for fold, (tr_idx, va_idx) in enumerate(folds):
    oof_probs, test_probs, f1 = train_bert_fold(tr_idx, va_idx, fold)
    bert_oof[va_idx] = oof_probs
    bert_test += test_probs / N_FOLDS
    bert_scores.append(f1)

bert_oof_f1 = f1_score(labels, bert_oof.argmax(axis=1), average='macro')
bert_time = time.time() - t0

print(f'\nBERT OOF F1: {bert_oof_f1:.4f}')
print(f'BERT Mean Fold F1: {np.mean(bert_scores):.4f} +/- {np.std(bert_scores):.4f}')
print(f'BERT training time: {bert_time/60:.1f} min')
print(f'\n>>> GPU portion done! Rest is CPU only. <<<')

In [ ]:
# =============================================================================
# Cell 7: TF-IDF Features (CPU)
# =============================================================================

print('Building TF-IDF features...')

# Full text: word + char
tfidf_word = TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 6), min_df=3, max_df=0.95,
                              sublinear_tf=True, max_features=100000)

X_tr_word = tfidf_word.fit_transform(train_df['clean_tfidf'])
X_te_word = tfidf_word.transform(test_df['clean_tfidf'])
X_tr_char = tfidf_char.fit_transform(train_df['clean_tfidf'])
X_te_char = tfidf_char.transform(test_df['clean_tfidf'])

# Achados
tfidf_ach = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_tr_ach = tfidf_ach.fit_transform(train_df['achados_tfidf'])
X_te_ach = tfidf_ach.transform(test_df['achados_tfidf'])

X_train_sparse = hstack([X_tr_word, X_tr_char, X_tr_ach]).tocsr()
X_test_sparse  = hstack([X_te_word, X_te_char, X_te_ach]).tocsr()

# SVD para tree models
svd = TruncatedSVD(n_components=80, random_state=42)
X_tr_svd = svd.fit_transform(X_train_sparse)
X_te_svd = svd.transform(X_test_sparse)

# Tree features: SVD + dense
X_tr_tree = np.hstack([X_tr_svd, dense_train.values.astype(np.float32)])
X_te_tree = np.hstack([X_te_svd, dense_test.values.astype(np.float32)])

print(f'Sparse: {X_train_sparse.shape[1]}, Tree: {X_tr_tree.shape[1]}')

In [ ]:
# =============================================================================
# Cell 8: CPU Models — SVC + LightGBM + CatBoost
# =============================================================================

# --- SVC ---
print('Training SVC...')
svc_oof  = np.zeros((len(train_df), N_CLASSES))
svc_test = np.zeros((len(test_df), N_CLASSES))
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', C=1.0, random_state=42, max_iter=15000),
        cv=3, method='sigmoid'
    )
    m.fit(X_train_sparse[tr_idx], labels[tr_idx])
    svc_oof[va_idx] = m.predict_proba(X_train_sparse[va_idx])
    svc_test += m.predict_proba(X_test_sparse) / N_FOLDS
svc_f1 = f1_score(labels, svc_oof.argmax(1), average='macro')
print(f'  SVC OOF F1: {svc_f1:.4f}')

# --- LightGBM ---
print('Training LightGBM...')
lgb_oof  = np.zeros((len(train_df), N_CLASSES))
lgb_test = np.zeros((len(test_df), N_CLASSES))
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = lgb.LGBMClassifier(class_weight='balanced', n_estimators=500, learning_rate=0.03,
                            max_depth=7, num_leaves=63, subsample=0.8, colsample_bytree=0.6,
                            random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_tr_tree[tr_idx], labels[tr_idx])
    lgb_oof[va_idx] = m.predict_proba(X_tr_tree[va_idx])
    lgb_test += m.predict_proba(X_te_tree) / N_FOLDS
lgb_f1 = f1_score(labels, lgb_oof.argmax(1), average='macro')
print(f'  LGB OOF F1: {lgb_f1:.4f}')

# --- CatBoost ---
print('Training CatBoost...')
cb_oof  = np.zeros((len(train_df), N_CLASSES))
cb_test = np.zeros((len(test_df), N_CLASSES))
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=3,
        auto_class_weights='Balanced', random_seed=42, verbose=0, task_type='CPU'
    )
    m.fit(X_tr_tree[tr_idx], labels[tr_idx],
          eval_set=(X_tr_tree[va_idx], labels[va_idx]), early_stopping_rounds=50)
    cb_oof[va_idx] = m.predict_proba(X_tr_tree[va_idx])
    cb_test += m.predict_proba(X_te_tree) / N_FOLDS
cb_f1 = f1_score(labels, cb_oof.argmax(1), average='macro')
print(f'  CatBoost OOF F1: {cb_f1:.4f}')

print(f'\nAll CPU models done.')

In [ ]:
# =============================================================================
# Cell 9: Stacking Meta-Learner — BERT + SVC + LGB + CatBoost
# =============================================================================

print('='*60)
print('STACKING: 4 Models → Meta-Learner')
print('='*60)

print(f'  BERT:     {bert_oof_f1:.4f}')
print(f'  SVC:      {svc_f1:.4f}')
print(f'  LGB:      {lgb_f1:.4f}')
print(f'  CatBoost: {cb_f1:.4f}')

all_oof  = {'bert': bert_oof, 'svc': svc_oof, 'lgb': lgb_oof, 'catboost': cb_oof}
all_test = {'bert': bert_test, 'svc': svc_test, 'lgb': lgb_test, 'catboost': cb_test}
model_names = list(all_oof.keys())
n_models = len(model_names)

# Stack OOF: (n, 4*7) = 28 features
oof_stack = np.hstack([all_oof[n] for n in model_names])
te_stack  = np.hstack([all_test[n] for n in model_names])

# Meta-learner: LogReg
oof_meta = np.zeros((len(train_df), N_CLASSES))
te_meta  = np.zeros((len(test_df), N_CLASSES))
for fi, (tr_idx, va_idx) in enumerate(folds):
    meta = LogisticRegression(class_weight='balanced', C=1.0, max_iter=3000,
                              solver='lbfgs', multi_class='multinomial', random_state=42)
    meta.fit(oof_stack[tr_idx], labels[tr_idx])
    oof_meta[va_idx] = meta.predict_proba(oof_stack[va_idx])
    te_meta += meta.predict_proba(te_stack) / N_FOLDS
meta_f1 = f1_score(labels, oof_meta.argmax(1), average='macro')
print(f'\nMeta-learner OOF F1: {meta_f1:.4f}')

# Dirichlet weight search
print('Dirichlet weight search...')
best_f1, best_weights = 0, None
np.random.seed(42)
for _ in range(20000):
    raw = np.random.dirichlet(np.ones(n_models))
    w = np.round(raw / 0.05) * 0.05
    if w.sum() == 0: continue
    w /= w.sum()
    blend = sum(w[i] * all_oof[model_names[i]] for i in range(n_models))
    score = f1_score(labels, blend.argmax(1), average='macro')
    if score > best_f1:
        best_f1, best_weights = score, w.copy()

print(f'Dirichlet OOF F1: {best_f1:.4f}')
for n, w in zip(model_names, best_weights):
    if w > 0: print(f'  {n}: {w:.2f}')

oof_weighted = sum(best_weights[i] * all_oof[model_names[i]] for i in range(n_models))
te_weighted  = sum(best_weights[i] * all_test[model_names[i]] for i in range(n_models))

# Hybrid
oof_hybrid = 0.5 * oof_meta + 0.5 * oof_weighted
te_hybrid  = 0.5 * te_meta + 0.5 * te_weighted
hybrid_f1 = f1_score(labels, oof_hybrid.argmax(1), average='macro')
print(f'Hybrid OOF F1: {hybrid_f1:.4f}')

# Select best
results = {'meta': (meta_f1, oof_meta, te_meta),
           'weighted': (best_f1, oof_weighted, te_weighted),
           'hybrid': (hybrid_f1, oof_hybrid, te_hybrid)}
best_strategy = max(results, key=lambda k: results[k][0])
final_score, oof_blend, te_blend = results[best_strategy]
print(f'\nBest: {best_strategy} (F1={final_score:.4f})')

In [ ]:
# =============================================================================
# Cell 10: Threshold Tuning + Clinical Guardrails + Submission
# =============================================================================

threshold_classes = [6, 5, 4, 3, 1, 0]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in threshold_classes:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for hc in threshold_classes:
                if hc == cls: break
                if hc in thresholds: mask = mask & (preds != hc)
            preds[mask] = cls
    return preds

baseline_f1 = f1_score(labels, np.argmax(oof_blend, axis=1), average='macro')
print(f'Baseline OOF F1 (no thresholds): {baseline_f1:.4f}')

best_thresholds = {}
current_f1 = baseline_f1
for cls in threshold_classes:
    best_cls_f1, best_cls_thr = current_f1, None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_f1 = f1_score(labels, apply_thresholds(oof_blend, trial), average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1, best_cls_thr = trial_f1, thr
    if best_cls_thr:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f'Class {cls}: thr={best_cls_thr:.2f}, F1={best_cls_f1:.4f}')
    else:
        print(f'Class {cls}: no improvement')

final_oof_f1 = f1_score(labels, apply_thresholds(oof_blend, best_thresholds), average='macro')
print(f'\nFinal OOF macro-F1: {final_oof_f1:.4f}')
print(f'Thresholds: {best_thresholds}')
print(classification_report(labels, apply_thresholds(oof_blend, best_thresholds), digits=4))

# --- Submission ---
test_preds = apply_thresholds(te_blend, best_thresholds)
test_df['target'] = test_preds

def apply_clinical_guardrails(row):
    text = str(row['report']).lower()
    pred = int(row['target'])
    birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if birads_match:
        mentioned = int(birads_match.group(1))
        if 0 <= mentioned <= 6: return mentioned
    if re.search(r'resultado de cine grau 3|carcinoma|neoplasia maligna', text): return 6
    if re.search(r'n[oó]dulo\s+espiculad', text) and pred < 4: return 5
    if 'espiculad' in text and 'distor' in text and pred < 4: return 5
    if re.search(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', text):
        if pred >= 4 and not re.search(r'espiculad|suspeit|malign|distor', text): return 2
    return pred

test_df['target'] = test_df.apply(apply_clinical_guardrails, axis=1)

submission = test_df[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

total_time = time.time() - t0
print(f'\nSubmission saved!')
print(f'Total runtime: {total_time/60:.1f} min (BERT: {bert_time/60:.1f} min)')
print(f'Prediction distribution:\n{submission["target"].value_counts().sort_index()}')
print(submission)